In [44]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch



df = pd.read_csv('../../datasets/gekuerzt_bereinigt.csv', dtype=str)
print(df.head(10))
print('Alle SPalten', df.columns.tolist())

#▪ Concretely, implement and analyze:
#▪ Tokenizer (with “unknown” and “endoftext” handling)
#▪ BPE
#▪ Data sampling - try different context sizes and strides
#▪ Put it together (token embeddings) and integrate positional embeddings


  Unnamed: 0                                              title  \
0    2131142                     Vegetable Frittata with Cheese   
1    2131143  Grilled Clams with Spaghetti, Prosciutto, and ...   
2    2131144                          Spiced up Vegetable Pasta   
3    2131145           Secret Ingredient Chocolate Chip Cookies   
4    2131146                       Chantri's Poppy Seed Chicken   
5    2131147                            Southwestern Rice Salad   
6    2131148                                        Quesadillas   
7    2131149  Roasted Tomato and Shallot Salad with Goat Cheese   
8    2131150                Mixed Grain Sour Dough Bread Recipe   
9    2131151                              Italian Cheeseburgers   

                                         ingredients  \
0  ["8 large eggs slightly beaten", "1 tablespoon...   
1  ["1/2 cup extra-virgin olive oil", "4 garlic c...   
2  ["14 cup bell pepper, finely chopped (yellow, ...   
3  ["12 cup butter", "12 cup butter fl

Tokenizer: Hugging Face Tokenizer
-> Text wird bereinigt; Groß/Kleinschreibung
-> Voraufteilung (Pre-Tokenization) Text wird in Leerzeichen und Satzzeichen aufgeteilt
-> Häufige Zeichenpaare werden analysiert. Jeder Token bekommt eine ID

Doing: nur title; ingedients, directions und ner werden in den Tokenizer einzeln als Text eingefügt. Der Rest fällt auf Grund der irrelevanz weg.

In [ ]:
print(df.isna().sum())
print(df.apply(type).value_counts())
print(df.head(0))
print(df.apply(type).value_counts())


df = df[['title', 'ingredients', 'directions', 'NER']]
df = df[:100]

for col in ['ingredients', 'NER']:
    df[col] = df[col].apply(lambda x: ' '.join(eval(x)))


df['combined'] = df['title'] + ' ' + df['ingredients'] + ' ' + df['directions'] + ' ' + df['NER']

# Schritt 4: Tokenizer laden und anwenden
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

df['tokens'] = df['combined'].apply(lambda x: tokenizer.tokenize(x))
df['input_ids'] = df['combined'].apply(lambda x: tokenizer(x)['input_ids'])

print(df.head(10))

Unnamed: 0     0
title          0
ingredients    0
directions     0
link           0
source         0
NER            0
dtype: int64
<class 'pandas.core.series.Series'>    7
Name: count, dtype: int64
Empty DataFrame
Columns: [Unnamed: 0, title, ingredients, directions, link, source, NER]
Index: []
<class 'pandas.core.series.Series'>    7
Name: count, dtype: int64


Token indices sequence length is longer than the specified maximum sequence length for this model (618 > 512). Running this sequence through the model will result in indexing errors


                                               title  \
0                     Vegetable Frittata with Cheese   
1  Grilled Clams with Spaghetti, Prosciutto, and ...   
2                          Spiced up Vegetable Pasta   
3           Secret Ingredient Chocolate Chip Cookies   
4                       Chantri's Poppy Seed Chicken   
5                            Southwestern Rice Salad   
6                                        Quesadillas   
7  Roasted Tomato and Shallot Salad with Goat Cheese   
8                Mixed Grain Sour Dough Bread Recipe   
9                              Italian Cheeseburgers   

                                         ingredients  \
0  8 large eggs slightly beaten 1 tablespoon basi...   
1  1/2 cup extra-virgin olive oil 4 garlic cloves...   
2  14 cup bell pepper, finely chopped (yellow, or...   
3  12 cup butter 12 cup butter flavor shortening ...   
4  4 chicken breasts 1 can cream of mushroom soup...   
5  3-3/4 qt. cooked long-grain white rice 3-3/4

Data Sampling; Bedeutung: Aufteilung der input_ids in Fenster (Chunks) -> Tokenizer kann manximal 512 Token aufteilen -> Lösung: Aufteilung in Chunks

In [47]:
max_length = 512
stride = 256

all_chunks = []
all_embeddings = []
model = AutoModel.from_pretrained("bert-base-uncased")
for text in df['combined']:
    tokens = tokenizer(
        text,
        max_length=max_length,
        stride=stride,
        truncation=True,
        return_overflowing_tokens=True,
        padding="max_length",
        return_tensors="pt" 
    )
    all_chunks.extend(tokens['input_ids'])


    with torch.no_grad():
        output = model(**{k: v for k, v in tokens.items() if k != 'overflow_to_sample_mapping'})
    
    embedding = output.last_hidden_state[:, 0, :]
    all_embeddings.append(embedding)

print(f"Anzahl Chunks: {len(all_chunks)}")
print(f"Anzahl Embeddings: {len(all_embeddings)}")
print(f"Embedding Shape: {all_embeddings[0].shape}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20248.08it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 